In [3]:
import profilelib.*

In [4]:
val data = mapOf(
    "original" to load_data("../../jfrStorage/serialization-twitterBM-replication/sampleInterval_and_cycles_variance_measurement/"),
    "new_base_with_memory" to load_data("../../jfrStorage/serialization-twitterBM-replication/new_base_with_memory/"),
    "control" to load_data("../../jfrStorage/serialization-twitterBM-replication/control/"),
    "control2" to load_data("../../jfrStorage/serialization-twitterBM-replication/control2/"),
)

'../../jfrStorage/serialization-twitterBM-replication/sampleInterval_and_cycles_variance_measurement/platform=jvm,cycles=5000,sampleInterval=5,iteration=22.jfr'
'../../jfrStorage/serialization-twitterBM-replication/sampleInterval_and_cycles_variance_measurement/platform=native,cycles=100000,sampleInterval=5,iteration=9.jfr'
'../../jfrStorage/serialization-twitterBM-replication/sampleInterval_and_cycles_variance_measurement/platform=native,cycles=50000,sampleInterval=1,iteration=20.jfr'
'../../jfrStorage/serialization-twitterBM-replication/sampleInterval_and_cycles_variance_measurement/platform=jvm,cycles=100000,sampleInterval=1,iteration=19.jfr'
'../../jfrStorage/serialization-twitterBM-replication/sampleInterval_and_cycles_variance_measurement/platform=native,cycles=10000,sampleInterval=10,iteration=2.jfr'
'../../jfrStorage/serialization-twitterBM-replication/sampleInterval_and_cycles_variance_measurement/platform=native,cycles=500,sampleInterval=1,iteration=2.jfr'
'../../jfrStorage/s

In [11]:
compareVarianceAndBias(data.mapValues { (name, dataset) ->
    dataset
        .toList()
        .filter { (params, _) -> params["sampleInterval"] == "100" && params["cycles"] == "100000" }
        .groupBy { (params, _) -> params["iteration"]!! }
        .mapNotNull { (_, platformPair) ->
            val jvm = platformPair
                .singleOrNull { (params, _) -> params["platform"]!! == "jvm" }
                ?.let { (_, freqs) -> freqs }
                ?.getOrDefault("java.lang.String.substring", 0)
            val native = platformPair
                .singleOrNull { (params, _) -> params["platform"]!! == "native" }
                ?.let { (_, freqs) -> freqs }
                ?.getOrDefault("weirdo:Kotlin_String_subSequence", 0)
            if (jvm == null || native == null) null
            else jvm.toDouble() / native.toDouble()
        }
})

original: 0.952 (0.061)
new_base_with_memory: 0.956 (0.067)
control: 0.951 (0.060)
control2: 0.934 (0.067)

Bias: 0.022482567839812617 (0.0237959379477148)
